<a href="https://colab.research.google.com/github/RanaAAAli/bachelor-/blob/main/Copy_of_notebook71d6f1caa9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving EHR.csv to EHR (4).csv


In [ ]:
df = pd.read_csv('EHR.csv')

print("Initial Dataset Shape:", df.shape)
df.head()

Initial Dataset Shape: (1447, 29)


,patientunitstayid,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,wardid,apacheadmissiondx,admissionheight,hospitaladmittime24,...,unitadmitsource,unitvisitnumber,unitstaytype,admissionweight,dischargeweight,unitdischargetime24,unitdischargeoffset,unitdischargelocation,unitdischargestatus,uniquepid
0,210014,182373,Male,45,Caucasian,73,89,"Hypertension, uncontrolled (for cerebrovascula...",178.0,13:08:59,...,Direct Admit,1,admit,116.0,112.7,15:00:00,4424,Skilled Nursing Facility,Alive,002-10665
1,200026,174624,Male,50,Caucasian,71,87,Ablation or mapping of cardiac conduction pathway,177.8,10:41:00,...,Operating Room,1,admit,106.1,106.1,17:40:00,1548,Home,Alive,002-10715
2,221131,190993,Male,83,Caucasian,71,87,"Endarterectomy, carotid",175.3,21:43:00,...,Operating Room,1,admit,NaN,72.1,17:46:00,1203,Home,Alive,002-10249
3,221215,191054,Male,49,Caucasian,71,87,"Infarction, acute myocardial (MI)",185.4,03:16:00,...,Emergency Department,1,admit,145.3,146.6,19:07:00,1562,Home,Alive,002-10627
4,217835,188445,Male,57,Caucasian,73,92,"CABG alone, coronary artery bypass grafting",172.7,01:09:00,...,Operating Room,1,admit,NaN,80.4,08:25:00,4719,Floor,Alive,002-10324


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1447 entries, 0 to 1446
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   patientunitstayid          1447 non-null   int64  
 1   patienthealthsystemstayid  1447 non-null   int64  
 2   gender                     1444 non-null   object 
 3   age                        1446 non-null   object 
 4   ethnicity                  1405 non-null   object 
 5   hospitalid                 1447 non-null   int64  
 6   wardid                     1447 non-null   int64  
 7   apacheadmissiondx          1267 non-null   object 
 8   admissionheight            1402 non-null   float64
 9   hospitaladmittime24        1447 non-null   object 
 10  hospitaladmitoffset        1447 non-null   int64  
 11  hospitaladmitsource        1218 non-null   object 
 12  hospitaldischargeyear      1447 non-null   int64  
 13  hospitaldischargetime24    1447 non-null   objec

In [ ]:
print("Missing Values:\n", df.isnull().sum())
print("\nSummary Statistics:\n", df.describe())

Missing Values:
 patientunitstayid              0
patienthealthsystemstayid      0
gender                         3
age                            1
ethnicity                     42
hospitalid                     0
wardid                         0
apacheadmissiondx            180
admissionheight               45
hospitaladmittime24            0
hospitaladmitoffset            0
hospitaladmitsource          229
hospitaldischargeyear          0
hospitaldischargetime24        0
hospitaldischargeoffset        0
hospitaldischargelocation      8
hospitaldischargestatus        7
unittype                       0
unitadmittime24                0
unitadmitsource               19
unitvisitnumber                0
unitstaytype                   0
admissionweight              134
dischargeweight              576
unitdischargetime24            0
unitdischargeoffset            0
unitdischargelocation          5
unitdischargestatus            2
uniquepid                      0
dtype: int64

Summary Stat

In [ ]:
# Split dataset WITHOUT overlap
df_A = df.sample(frac=0.33, random_state=1)

remaining = df.drop(df_A.index)
df_B = remaining.sample(frac=0.5, random_state=2)

df_C = remaining.drop(df_B.index)

# Add hospital labels
df_A = df_A.copy()
df_B = df_B.copy()
df_C = df_C.copy()

df_A['Hospital'] = 'Hospital_A'
df_B['Hospital'] = 'Hospital_B'
df_C['Hospital'] = 'Hospital_C'

In [ ]:
# Always work on copies to avoid warnings
df_B.loc[:, 'gender'] = df_B['gender'].replace({'Male': 'M', 'Female': 'F'})
df_C.loc[:, 'gender'] = df_C['gender'].astype(str).str.lower()

# Add missing values safely
sample_idx = df_A.sample(frac=0.1, random_state=1).index
df_A.loc[sample_idx, 'age'] = np.nan

# Diagnosis inconsistency (if column exists)
if 'apacheadmissiondx' in df_C.columns:
    df_C.loc[:, 'apacheadmissiondx'] = df_C['apacheadmissiondx'].astype(str).str.upper()

In [ ]:
df_fragmented = pd.concat([df_A, df_B, df_C], ignore_index=True)

print("Fragmented Dataset Shape:", df_fragmented.shape)

Fragmented Dataset Shape: (1447, 30)


In [ ]:
import pandas as pd

# Standardize gender
df_fragmented['gender'] = df_fragmented['gender'].replace({
    'M': 'Male',
    'F': 'Female',
    'male': 'Male',
    'female': 'Female'
})

# Fix age column using pd.to_numeric to handle non-numeric strings
if 'age' in df_fragmented.columns:
    df_fragmented['age'] = pd.to_numeric(df_fragmented['age'], errors='coerce')
    # Now we can safely calculate the median and fill missing values
    median_age = df_fragmented['age'].median()
    df_fragmented['age'] = df_fragmented['age'].fillna(median_age)

# Standardize diagnosis
if 'apacheadmissiondx' in df_fragmented.columns:
    df_fragmented['apacheadmissiondx'] = df_fragmented['apacheadmissiondx'].astype(str).str.lower()

In [ ]:
# Explicitly use .copy() to avoid SettingWithCopyWarning later
if 'patienthealthsystemstayid' in df_fragmented.columns:
    df_cleaned = df_fragmented.drop_duplicates(subset=['patienthealthsystemstayid']).copy()
else:
    df_cleaned = df_fragmented.copy()

print("Cleaned Dataset Shape:", df_cleaned.shape)

Cleaned Dataset Shape: (1242, 30)


In [ ]:
if all(col in df_cleaned.columns for col in ['hospitaladmittime24', 'hospitaldischargetime24']):
    # Convert columns to datetime to allow for subtraction
    df_cleaned['hospitaladmittime24'] = pd.to_datetime(df_cleaned['hospitaladmittime24'], format='%H:%M:%S', errors='coerce')
    df_cleaned['hospitaldischargetime24'] = pd.to_datetime(df_cleaned['hospitaldischargetime24'], format='%H:%M:%S', errors='coerce')

    # Calculate length of stay
    df_cleaned['length_of_stay'] = (
        df_cleaned['hospitaldischargetime24'] - df_cleaned['hospitaladmittime24']
    )